# **Part 1: Setup & Data Loading**

In [1]:
pip install --quiet datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
from datasets import Dataset
from datasets import load_dataset

In [4]:
# Load the UCI SMS Spam dataset from Hugging Face hub
df = pd.read_parquet("hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet")

# Convert df to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# TODO: Split the data - use 4,000 samples for training and 1,000 for validation
train_ds = hf_dataset.select(range(4000))
val_ds   = hf_dataset.select(range(4000, 5000))

# Display the first few rows to understand the data structure
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


# **Part 2: Tokenization Setup**

In [5]:
from transformers import GPT2Tokenizer

model_name = "gpt2"

tokenizer  = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 has no pad token by default--set it to eos

tokenizer.pad_token = "eos" # TODO: set the pad token to the eos token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [7]:
def tokenize_fn(examples):

    # TODO: Complete the tokenization function

    return tokenizer(

        examples["sms"],

        padding="max_length",

        truncation= True,

        max_length=64

    )

# Apply tokenization to both datasets

train_tok = train_ds.map(tokenize_fn, batched=True)

val_tok   = val_ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# Part 3: Pre-trained Model Setup

In [14]:
import torch

from transformers import GPT2ForSequenceClassification

model = GPT2ForSequenceClassification.from_pretrained(

    pretrained_model_name_or_path= "gpt2",

    num_labels= 2,

    pad_token_id= tokenizer.eos_token_id

)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# **Part 4: Custom Attention Implementation**

In [15]:
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import torch.nn as nn

class Attention(nn.Module):

    def __init__(self, embed_dim):

        super().__init__()

        # TODO: Initialize the scaling factor (hint: embed_dim ** -0.5)

        self.scale = embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):

        # TODO: Calculate attention scores using matrix multiplication

        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale



        if mask is not None:

            scores = scores.masked_fill(mask == 0, float('-inf'))



        # TODO: Apply softmax to get attention weights

        attn = F.softmax(scores, dim=-1)



        # TODO: Apply attention weights to values

        return torch.matmul(attn, value), attn

In [18]:
class SimpleAttentionClassifier(nn.Module):

    def __init__(self, vocab_size, embed_dim, num_classes):

        super().__init__()

        # TODO: Create embedding layer

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # TODO: Initialize attention layer

        self.attn = Attention(embed_dim)

        # TODO: Create final classification layer

        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):

        # TODO: Get embeddings from input

        embed = self.embedding(x)

        # TODO: Apply self-attention (query=key=value=embed)

        attn_output, _ = self.attn(embed, embed, embed)

        # TODO: Pool the attention output (use mean over sequence dimension)

        pooled = attn_output.mean(dim=1)

        # TODO: Apply final linear layer

        return self.fc(pooled)

In [20]:
def preprocess_for_attention(example):

    # TODO: Encode text using tokenizer with proper parameters

    tokens = tokenizer.encode(

        example["sms"],

        max_length=64,

        truncation=True,

        padding="max_length"

    )

    return {"input_ids": tokens, "label": example["label"]}

# TODO: Apply preprocessing to both datasets

train_ds_attn = train_ds.map(preprocess_for_attention)

val_ds_attn = val_ds.map(preprocess_for_attention)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [21]:
class SMSDataset(Dataset):

    def __init__(self, hf_dataset):

        self.data = hf_dataset

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        item = self.data[idx]

        return {

            'input_ids': torch.tensor(item["input_ids"], dtype=torch.long),

            'label': torch.tensor(item["label"], dtype=torch.long)

        }

# TODO: Create data loaders

train_loader = DataLoader(SMSDataset(train_ds_attn), batch_size=32, shuffle=True)

val_loader = DataLoader(SMSDataset(val_ds_attn), batch_size=32)

In [22]:
# Setup training parameters

vocab_size = len(tokenizer) # TODO: get vocab_size from tokenizer

embed_dim = 64

num_classes = 2 # TODO: set for binary classification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TODO: Initialize the model

attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)

# TODO: Setup optimizer

optimizer = torch.optim.Adam(attn_model.parameters(), lr=1e-3) # TODO: set learning rate to 1e-3

# TODO: Setup loss function

criterion = nn.CrossEntropyLoss()

# Training loop

attn_model.train()

for batch in train_loader:

    inputs = batch['input_ids'].to(device)

    labels = batch['label'].to(device)

    optimizer.zero_grad()

    outputs = attn_model(inputs)

    loss = criterion(outputs, labels)

    loss.backward()

    optimizer.step()

print("Custom Attention model trained on SMS dataset. Sample batch loss:", loss.item())

Custom Attention model trained on SMS dataset. Sample batch loss: 0.2050466239452362


# **Part 5: Metrics & Evaluation**

In [23]:
import evaluate

import numpy as np

# TODO: Load evaluation metrics

accuracy  = evaluate.load("accuracy")

precision = evaluate.load("precision")

recall    = evaluate.load("recall")

f1        = evaluate.load("f1")

def compute_metrics(pred):

    logits, labels = pred

    preds = np.argmax(logits, axis=-1)

    return {

        "accuracy":  accuracy.compute(predictions=preds, references=labels)["accuracy"],

        # TODO: Compute precision using the same pattern as accuracy

        "precision": precision.compute(predictions=preds, references=labels)["precision"],

        # TODO: Compute recall using the same pattern as accuracy

        "recall":    recall.compute(predictions=preds, references=labels)["recall"],

        # TODO: Compute F1 using the same pattern as accuracy

        "f1":        f1.compute(predictions=preds, references=labels)["f1"]

    }

In [25]:
print("\n📊 Evaluating GPT-2 Model...")

gpt2_preds = []

gpt2_labels = []

model.eval()

for ex in val_tok:

    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)

    with torch.no_grad():

        logits = model(inputs).logits

    pred = torch.argmax(logits, dim=-1).cpu().item()

    gpt2_preds.append(pred)

    gpt2_labels.append(ex['label'])

# TODO: Compute GPT-2 metrics using the same pattern as in compute_metrics

gpt2_metrics = {

    "accuracy":  accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)["accuracy"],

    "precision": precision.compute(predictions=gpt2_preds, references=gpt2_labels)["precision"],

    "recall":    recall.compute(predictions=gpt2_preds, references=gpt2_labels)["recall"],

    "f1":        f1.compute(predictions=gpt2_preds, references=gpt2_labels)["f1"]

}

print("GPT-2 Metrics:", gpt2_metrics)

print("\n📊 Evaluating Custom Attention Model...")

attn_preds = []

attn_labels = []

attn_model.eval()

for batch in val_loader:

    inputs = batch['input_ids'].to(device)

    labels = batch['label'].to(device)

    with torch.no_grad():

        outputs = attn_model(inputs)

        preds = torch.argmax(outputs, dim=1)

    attn_preds.extend(preds.cpu().tolist())

    attn_labels.extend(labels.cpu().tolist())

# TODO: Compute attention model metrics using the same pattern

attn_metrics = {

    "accuracy":  accuracy.compute(predictions=attn_preds, references=attn_labels)["accuracy"],

    "precision": precision.compute(predictions=attn_preds, references=attn_labels)["precision"],

    "recall":    recall.compute(predictions=attn_preds, references=attn_labels)["recall"],

    "f1":        f1.compute(predictions=attn_preds, references=attn_labels)["f1"]

}

print("Attention Model Metrics:", attn_metrics)


📊 Evaluating GPT-2 Model...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


GPT-2 Metrics: {'accuracy': 0.861, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

📊 Evaluating Custom Attention Model...
Attention Model Metrics: {'accuracy': 0.861, 'precision': 0.5, 'recall': 0.007194244604316547, 'f1': 0.014184397163120567}


## **Part 6: Reflection Questions**

1. Roles of Query, Key, and Value

The query is what’s looking for information,
the key tells what each word represents,
and the value holds the actual information.
The model compares the query with all keys to find which values are most useful.

2. Why We Use a Scaling Factor (1/√dₖ)

When numbers get too big, the model starts giving too much attention to one word.
The scaling factor keeps things balanced so the model spreads attention more fairly and trains better.

3. How Self-Attention Differs from RNNs

RNNs read one word at a time, but self-attention looks at all words at once.
This helps it remember long sentences better, work faster, and understand context more clearly.

4. Performance Analysis

The transformer model performs better because it understands context and relationships between words.
The downside is that it needs more memory and power.
To improve it, we can add more attention heads or train it with more data.